### Import Dependencies 

In [ ]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

from dotenv import load_dotenv #import 
import os 

load_dotenv("../../.env")



### Download all data from qdrant

In [ ]:
 qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
all_points = qdrant_client.scroll(
     collection_name="Amazon-items-collection-01", 
     limit=100, 
     offset=None, 
     with_payload=True,
     with_vectors=False
)

In [ ]:
all_points
# all_points[0][0].payload

In [ ]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in all_points[0]]

In [ ]:
all_context # list of length 50 holds id of product and text which is preprocessed_description

### Render a prompt to generate synthetic Eval refernce dataset

In [ ]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""






In [ ]:
print(SYSTEM_PROMPT)
print(USER_PROMPT)

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    reasoning_effort='none'
)
print(response.choices[0].message.content)

In [ ]:
print(response.choices[0].message.content)

### so we took the qdrant data and ask the llm to give us questions that would yield more than 1, 1, no chunk results from the vector db 

In [ ]:
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [ ]:
print(json_output)

### create a dataset in langsmith...


In [ ]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item['chunk_ids']) == 1:
        single_chunk_counter += 1
    elif len(item['chunk_ids']) > 1:
        multiple_chunk_counter += 1
    elif len(item['chunk_ids']) == 0:
        impossible_counter += 1

print("Single chunk count:", single_chunk_counter)
print("Multiple chunk count:", multiple_chunk_counter)
print("Impossible count:", impossible_counter)

In [ ]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0BZ44Y4C6")
            )
        ]
    )
)

In [ ]:
def get_point_by_parent_asin(parent_asin):
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]
    return points[0].payload["preprocessed_description"]

In [ ]:
print(get_point_by_parent_asin("B0BZ44Y4C6"))

### Create eval set in Langsmith

In [ ]:
ls_client = Client()

dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG models"
)




In [ ]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_point_by_parent_asin(id) for id in item["chunk_ids"]]
        }
    )